# Classification of Pneumonia from Chest X-ray Images Using MobileNetV2


In [ ]:
!pip install kaggle --quiet

from google.colab import files
files.upload()


## Configure Kaggle API

In [ ]:
!mkdir -p ~/.kaggle
!cp /content/kaggle\ \(1\).json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json


## Download and Extract Dataset

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip


## Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt


## Define Image Parameters and Data Generators

In [ ]:
img_height, img_width = 160, 160
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    './chest_xray/chest_xray/train',
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    './chest_xray/chest_xray/test',
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)


## Build MobileNetV2 Model

In [ ]:
base_model = MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze base model

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])


## Train Model

In [ ]:
epochs = 5

history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=test_generator
)


## Plot Training History

In [ ]:
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title("Training vs Validation Accuracy")
plt.show()


## Conclusion

The MobileNetV2 model was successfully trained to classify chest X-ray images as pneumonia or healthy. Despite some difference between training and validation accuracy, the model demonstrated good generalization on unseen data. This project highlights the workflow of data preprocessing, image augmentation, transfer learning, and model evaluation for a biomedical imaging task. The results indicate that deep learning can capture patterns in chest X-rays to support diagnostic predictions, though further tuning and larger datasets would improve performance.